In [1]:
"""
Kaggle ML Toolkit - Production-Grade Utility Functions
=======================================================
A reusable utility script for machine learning pipelines on Kaggle.
Covers data profiling, preprocessing, model training, evaluation, and reporting.

Author: Dean Batur
Usage: Add this notebook as a utility script, then import in other notebooks:
    from kaggle_ml_toolkit import *
"""

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# 1. DATA PROFILING
# ============================================================

def data_profile(df, name="Dataset"):
    """
    Generate a concise data quality profile for a DataFrame.
    
    Parameters
    ----------
    df : pd.DataFrame
    name : str, optional
        Label for display.
    
    Returns
    -------
    pd.DataFrame
        Profile summary with dtype, missing counts, unique counts, and sample values.
    """
    profile = pd.DataFrame({
        "dtype": df.dtypes,
        "non_null": df.notnull().sum(),
        "null_count": df.isnull().sum(),
        "null_pct": (df.isnull().sum() / len(df) * 100).round(2),
        "nunique": df.nunique(),
        "sample": df.iloc[0] if len(df) > 0 else None
    })
    print(f"--- {name} Profile: {df.shape[0]} rows x {df.shape[1]} cols ---")
    return profile


def detect_feature_types(df, cardinality_threshold=20):
    """
    Automatically classify columns into numeric, categorical, binary, 
    high-cardinality, and datetime types.
    
    Parameters
    ----------
    df : pd.DataFrame
    cardinality_threshold : int
        Max unique values for a column to be considered categorical.
    
    Returns
    -------
    dict
        Dictionary with keys: numeric, categorical, binary, high_cardinality, datetime.
    """
    types = {
        "numeric": [],
        "categorical": [],
        "binary": [],
        "high_cardinality": [],
        "datetime": []
    }
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            types["datetime"].append(col)
        elif df[col].nunique() == 2:
            types["binary"].append(col)
        elif pd.api.types.is_numeric_dtype(df[col]):
            types["numeric"].append(col)
        elif df[col].nunique() <= cardinality_threshold:
            types["categorical"].append(col)
        else:
            types["high_cardinality"].append(col)
    return types


# ============================================================
# 2. PREPROCESSING
# ============================================================

def handle_missing(df, numeric_strategy="median", categorical_strategy="mode"):
    """
    Fill missing values with configurable strategies.
    
    Parameters
    ----------
    df : pd.DataFrame
    numeric_strategy : str
        One of 'mean', 'median', 'zero'.
    categorical_strategy : str
        One of 'mode', 'unknown'.
    
    Returns
    -------
    pd.DataFrame
        DataFrame with missing values filled.
    """
    df = df.copy()
    for col in df.columns:
        if df[col].isnull().sum() == 0:
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            if numeric_strategy == "mean":
                df[col].fillna(df[col].mean(), inplace=True)
            elif numeric_strategy == "median":
                df[col].fillna(df[col].median(), inplace=True)
            elif numeric_strategy == "zero":
                df[col].fillna(0, inplace=True)
        else:
            if categorical_strategy == "mode":
                mode_val = df[col].mode()
                df[col].fillna(mode_val[0] if len(mode_val) > 0 else "Unknown", inplace=True)
            elif categorical_strategy == "unknown":
                df[col].fillna("Unknown", inplace=True)
    return df


def cap_outliers(df, cols=None, method="iqr", factor=1.5):
    """
    Cap outliers using IQR or percentile method.
    
    Parameters
    ----------
    df : pd.DataFrame
    cols : list, optional
        Columns to process. Defaults to all numeric columns.
    method : str
        'iqr' or 'percentile'.
    factor : float
        IQR multiplier (default 1.5) or percentile bounds (e.g., 0.01 for 1st/99th).
    
    Returns
    -------
    pd.DataFrame
    """
    df = df.copy()
    if cols is None:
        cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    for col in cols:
        if method == "iqr":
            q1 = df[col].quantile(0.25)
            q3 = df[col].quantile(0.75)
            iqr = q3 - q1
            lower, upper = q1 - factor * iqr, q3 + factor * iqr
        elif method == "percentile":
            lower = df[col].quantile(factor)
            upper = df[col].quantile(1 - factor)
        df[col] = df[col].clip(lower, upper)
    return df


def encode_categoricals(df, cols=None, method="label"):
    """
    Encode categorical columns using label or one-hot encoding.
    
    Parameters
    ----------
    df : pd.DataFrame
    cols : list, optional
        Columns to encode. Defaults to object/category columns.
    method : str
        'label' or 'onehot'.
    
    Returns
    -------
    pd.DataFrame
    """
    df = df.copy()
    if cols is None:
        cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
    
    if method == "label":
        from sklearn.preprocessing import LabelEncoder
        for col in cols:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
    elif method == "onehot":
        df = pd.get_dummies(df, columns=cols, drop_first=True)
    return df


# ============================================================
# 3. MODEL TRAINING & CROSS-VALIDATION
# ============================================================

def train_and_evaluate(X, y, models, cv=5, task="classification", scoring=None):
    """
    Train multiple models with cross-validation and return comparison results.
    
    Parameters
    ----------
    X : pd.DataFrame or np.ndarray
        Feature matrix.
    y : pd.Series or np.ndarray
        Target variable.
    models : dict
        Dictionary of {name: estimator} pairs.
    cv : int
        Number of cross-validation folds.
    task : str
        'classification' or 'regression'.
    scoring : str, optional
        Sklearn scoring metric. Defaults to 'accuracy'/'r2' based on task.
    
    Returns
    -------
    pd.DataFrame
        Comparison table with mean and std scores for each model.
    """
    from sklearn.model_selection import cross_val_score
    
    if scoring is None:
        scoring = "accuracy" if task == "classification" else "r2"
    
    results = []
    for name, model in models.items():
        scores = cross_val_score(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
        results.append({
            "model": name,
            "mean_score": round(scores.mean(), 4),
            "std_score": round(scores.std(), 4),
            "min_score": round(scores.min(), 4),
            "max_score": round(scores.max(), 4),
        })
        print(f"  {name}: {scores.mean():.4f} (+/- {scores.std():.4f})")
    
    return pd.DataFrame(results).sort_values("mean_score", ascending=False).reset_index(drop=True)


def get_default_classifiers():
    """
    Return a dictionary of four standard classifiers for quick benchmarking.
    
    Returns
    -------
    dict
        {name: estimator} pairs for Logistic Regression, Random Forest, 
        Gradient Boosting, and XGBoost.
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
    
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    }
    try:
        from xgboost import XGBClassifier
        models["XGBoost"] = XGBClassifier(
            n_estimators=100, use_label_encoder=False,
            eval_metric="logloss", random_state=42, verbosity=0
        )
    except ImportError:
        pass
    return models


def get_default_regressors():
    """
    Return a dictionary of four standard regressors for quick benchmarking.
    
    Returns
    -------
    dict
        {name: estimator} pairs for Linear Regression, Random Forest, 
        Gradient Boosting, and XGBoost.
    """
    from sklearn.linear_model import Ridge
    from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
    
    models = {
        "Ridge Regression": Ridge(alpha=1.0, random_state=42),
        "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
        "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    }
    try:
        from xgboost import XGBRegressor
        models["XGBoost"] = XGBRegressor(
            n_estimators=100, random_state=42, verbosity=0
        )
    except ImportError:
        pass
    return models


# ============================================================
# 4. EVALUATION & REPORTING
# ============================================================

def classification_report_df(y_true, y_pred, target_names=None):
    """
    Generate sklearn classification report as a DataFrame.
    
    Parameters
    ----------
    y_true : array-like
    y_pred : array-like
    target_names : list, optional
    
    Returns
    -------
    pd.DataFrame
    """
    from sklearn.metrics import classification_report
    report = classification_report(y_true, y_pred, target_names=target_names, output_dict=True)
    return pd.DataFrame(report).T.round(4)


def regression_metrics(y_true, y_pred):
    """
    Compute standard regression metrics.
    
    Parameters
    ----------
    y_true : array-like
    y_pred : array-like
    
    Returns
    -------
    dict
        MAE, RMSE, R2, and MAPE.
    """
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    
    metrics = {
        "MAE": round(mean_absolute_error(y_true, y_pred), 4),
        "RMSE": round(np.sqrt(mean_squared_error(y_true, y_pred)), 4),
        "R2": round(r2_score(y_true, y_pred), 4),
    }
    # MAPE (handle zero values)
    mask = y_true != 0
    if mask.sum() > 0:
        metrics["MAPE"] = round(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100, 2)
    return metrics


def feature_importance_df(model, feature_names):
    """
    Extract feature importances from a fitted tree-based model as a sorted DataFrame.
    
    Parameters
    ----------
    model : fitted estimator
        Must have `feature_importances_` attribute.
    feature_names : list
    
    Returns
    -------
    pd.DataFrame
        Sorted by importance descending.
    """
    importance = pd.DataFrame({
        "feature": feature_names,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False).reset_index(drop=True)
    importance["cumulative"] = importance["importance"].cumsum()
    return importance


def deployment_readiness_summary(model_name, score, metric, threshold=None):
    """
    Print a deployment readiness summary for a trained model.
    
    Parameters
    ----------
    model_name : str
    score : float
    metric : str
    threshold : float, optional
        Minimum acceptable score for deployment.
    """
    status = "READY" if (threshold is None or score >= threshold) else "NOT READY"
    print("=" * 60)
    print("DEPLOYMENT READINESS SUMMARY")
    print("=" * 60)
    print(f"  Model           : {model_name}")
    print(f"  Primary Metric  : {metric}")
    print(f"  Score            : {score:.4f}")
    if threshold:
        print(f"  Threshold        : {threshold:.4f}")
    print(f"  Status           : {status}")
    print("=" * 60)
    return {"model": model_name, "metric": metric, "score": score, "status": status}


# ============================================================
# 5. CONVENIENCE UTILITIES
# ============================================================

def reduce_mem_usage(df, verbose=True):
    """
    Reduce DataFrame memory usage by downcasting numeric types.
    
    Parameters
    ----------
    df : pd.DataFrame
    verbose : bool
    
    Returns
    -------
    pd.DataFrame
    """
    start_mem = df.memory_usage(deep=True).sum() / 1024 ** 2
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object and str(col_type) != "category":
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type).startswith("int"):
                for dtype in [np.int8, np.int16, np.int32, np.int64]:
                    if c_min >= np.iinfo(dtype).min and c_max <= np.iinfo(dtype).max:
                        df[col] = df[col].astype(dtype)
                        break
            else:
                for dtype in [np.float16, np.float32, np.float64]:
                    if c_min >= np.finfo(dtype).min and c_max <= np.finfo(dtype).max:
                        df[col] = df[col].astype(dtype)
                        break
    end_mem = df.memory_usage(deep=True).sum() / 1024 ** 2
    if verbose:
        print(f"Memory: {start_mem:.2f} MB -> {end_mem:.2f} MB ({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)")
    return df


def quick_split(df, target_col, test_size=0.2, random_state=42):
    """
    Quick train/test split returning X_train, X_test, y_train, y_test.
    
    Parameters
    ----------
    df : pd.DataFrame
    target_col : str
    test_size : float
    random_state : int
    
    Returns
    -------
    tuple
        (X_train, X_test, y_train, y_test)
    """
    from sklearn.model_selection import train_test_split
    X = df.drop(columns=[target_col])
    y = df[target_col]
    return train_test_split(X, y, test_size=test_size, random_state=random_state)


def timer(func):
    """Decorator to time function execution."""
    import time
    from functools import wraps
    
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"  [{func.__name__}] completed in {elapsed:.2f}s")
        return result
    return wrapper
